# ML_1 — IoT Sensor Telemetry Data Cleaning & Normalization

**Tech stack:** Python, Pandas, NumPy only.

This notebook loads IoT temperature and humidity readings, validates the data, handles missing and invalid readings, optionally reduces short-lived spikes using a rolling median, applies Min-Max normalization, and saves the results.

Run cells from top to bottom. Sample data is included, so the notebook runs immediately.

## 1. Import libraries

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

print("Libraries imported successfully.")


## 2. Project configuration

These are generic example sensor limits. Change them to match your actual sensor specifications. The rolling median window is set to 3 for demonstration; set it to `None` to disable smoothing.

In [ ]:
SENSOR_BOUNDS = {
    "temperature": (-40.0, 85.0),  # Celsius
    "humidity": (0.0, 100.0)       # Percentage
}

ROLLING_MEDIAN_WINDOW = 3  # Use None to disable smoothing
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration ready.")


## 3. Create sample IoT data

The sample includes missing readings, an invalid humidity reading (150%), and a short-lived temperature spike.

In [ ]:
raw_df = pd.DataFrame({
    "timestamp": [
        "2026-01-01 09:00:00", "2026-01-01 09:01:00",
        "2026-01-01 09:02:00", "2026-01-01 09:03:00",
        "2026-01-01 09:04:00", "2026-01-01 09:05:00",
        "2026-01-01 09:06:00", "2026-01-01 09:07:00",
        "2026-01-01 09:08:00", "2026-01-01 09:09:00"
    ],
    "temperature": [21.4, 21.6, np.nan, 22.0, 22.2, 22.4, 50.0, 22.8, 23.0, 23.2],
    "humidity":    [48.2, 48.5, 48.7, 49.1, np.nan, 49.8, 150.0, 50.4, 50.8, 51.1]
})

print("Raw dataset:")
display(raw_df)


### Optional: use your own CSV

Your file must contain `timestamp`, `temperature`, and `humidity`. Run this cell only when you want to upload your own CSV; it replaces the sample `raw_df`.

In [ ]:
# Uncomment these lines in Google Colab to upload your own CSV:
# from google.colab import files
# uploaded = files.upload()
# csv_filename = next(iter(uploaded))
# raw_df = pd.read_csv(csv_filename)
# display(raw_df.head())


## 4. Validate and prepare the data

In [ ]:
def prepare_telemetry(dataframe):
    required_columns = {"timestamp", "temperature", "humidity"}
    missing_columns = required_columns - set(dataframe.columns)

    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    data = dataframe.copy()
    data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
    data["temperature"] = pd.to_numeric(data["temperature"], errors="coerce")
    data["humidity"] = pd.to_numeric(data["humidity"], errors="coerce")

    invalid_timestamp_count = int(data["timestamp"].isna().sum())
    data = data.dropna(subset=["timestamp"])
    data = data.sort_values("timestamp").reset_index(drop=True)

    if data.empty:
        raise ValueError("No rows with valid timestamps were found.")

    return data, invalid_timestamp_count


prepared_df, invalid_timestamp_count = prepare_telemetry(raw_df)

print("Prepared data:")
display(prepared_df)


## 5. Clean missing and invalid sensor readings

Values outside the configured physical limits are marked as missing. Missing readings are linearly interpolated after sorting by timestamp. This row-based interpolation assumes readings are collected at roughly regular intervals.

In [ ]:
def clean_telemetry(
    dataframe,
    sensor_bounds,
    invalid_timestamps_dropped=0,
    rolling_median_window=None
):
    cleaned = dataframe.copy()
    report = {
        "rows_received": int(len(cleaned)),
        "invalid_timestamps_dropped": int(invalid_timestamps_dropped),
        "invalid_values_by_column": {},
        "missing_before_interpolation": {},
        "missing_after_interpolation": {},
        "rolling_median_window": rolling_median_window
    }

    for column, (lower_limit, upper_limit) in sensor_bounds.items():
        values = pd.to_numeric(cleaned[column], errors="coerce")

        outside_limits = values.notna() & (
            (values < lower_limit) | (values > upper_limit)
        )

        report["invalid_values_by_column"][column] = int(outside_limits.sum())
        values = values.mask(outside_limits)

        report["missing_before_interpolation"][column] = int(values.isna().sum())
        values = values.interpolate(method="linear", limit_direction="both")

        if values.isna().all():
            raise ValueError(f"Column {column} has no valid readings to interpolate.")

        cleaned[column] = values
        report["missing_after_interpolation"][column] = int(values.isna().sum())

    if rolling_median_window is not None:
        if rolling_median_window < 3 or rolling_median_window % 2 == 0:
            raise ValueError("Rolling median window must be an odd integer >= 3.")

        for column in sensor_bounds:
            cleaned[column] = cleaned[column].rolling(
                window=rolling_median_window,
                center=True,
                min_periods=1
            ).median()

    report["rows_after_cleaning"] = int(len(cleaned))
    return cleaned, report


cleaned_df, cleaning_report = clean_telemetry(
    prepared_df,
    SENSOR_BOUNDS,
    invalid_timestamps_dropped=invalid_timestamp_count,
    rolling_median_window=ROLLING_MEDIAN_WINDOW
)

print("Cleaned sensor data:")
display(cleaned_df)


## 6. Apply Min-Max normalization using NumPy

Formula: `scaled = (value - minimum) / (maximum - minimum)`

The scaled columns should be between 0 and 1. A constant column is mapped to 0.

In [ ]:
def minmax_normalize(dataframe, columns=("temperature", "humidity")):
    normalized = dataframe.copy()
    scaling_parameters = {}

    for column in columns:
        values = normalized[column].to_numpy(dtype=float)

        if not np.isfinite(values).all():
            raise ValueError(f"{column} still contains missing or non-finite values.")

        minimum = float(np.min(values))
        maximum = float(np.max(values))
        value_range = maximum - minimum

        if np.isclose(value_range, 0.0):
            scaled_values = np.zeros_like(values, dtype=float)
        else:
            scaled_values = (values - minimum) / value_range

        normalized[f"{column}_scaled"] = scaled_values
        scaling_parameters[column] = {"min": minimum, "max": maximum}

    return normalized, scaling_parameters


final_df, scaling_parameters = minmax_normalize(cleaned_df)

print("Normalized dataset:")
display(final_df)


## 7. Validate the results

In [ ]:
print("Scaled feature ranges:")
display(final_df[["temperature_scaled", "humidity_scaled"]].agg(["min", "max"]))

assert final_df[["temperature", "humidity"]].isna().sum().sum() == 0
assert final_df["temperature_scaled"].between(0, 1).all()
assert final_df["humidity_scaled"].between(0, 1).all()

print("Validation passed: no missing sensor values; scaled values are within [0, 1].")


## 8. Save the cleaned dataset and processing report

In [ ]:
csv_output_path = OUTPUT_DIR / "cleaned_telemetry.csv"
json_output_path = OUTPUT_DIR / "processing_report.json"

final_df.to_csv(csv_output_path, index=False)

processing_report = {
    **cleaning_report,
    "scaling_parameters": scaling_parameters,
    "output_rows": int(len(final_df)),
    "output_columns": list(final_df.columns)
}

with open(json_output_path, "w", encoding="utf-8") as report_file:
    json.dump(processing_report, report_file, indent=4)

print("Pipeline completed successfully.")
print(f"Cleaned CSV saved to: {csv_output_path}")
print(f"Processing report saved to: {json_output_path}")


## 9. View the processing report

In [ ]:
print(json.dumps(processing_report, indent=4))


## 10. Download the output files from Google Colab

Run this last cell when you want to download the generated CSV and JSON files.

In [ ]:
from google.colab import files

files.download(str(csv_output_path))
files.download(str(json_output_path))
